# Week 3 · Day 4: Logistic loss and cost

**Course:** Labs 04–05: Loss and cost

The teaching question is **“Is this source's absolute G magnitude below 4?”** Class 1 means yes. This is a brightness label, not a verified stellar type. Only `bp_rp` (color) enters the model; magnitude and parallax are used to construct the answer and must stay out of the inputs.

Day 1 saves a source-disjoint 60/20/20 split. Days 2–6 explore training and validation; Day 7 evaluates the reserved test set. Absolute magnitudes are uncorrected for dust, and the archive sample may not represent all Gaia sources.

## Setup

[Prerequisites and setup explanation](week3_prerequisites.ipynb). Run cells from top to bottom. Run Day 1 first. This notebook reads its saved data.

In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd
from IPython.display import display

# Jupyter may start in the project folder or in its notebooks folder.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src/classification.py').is_file():
    raise RuntimeError('Open Jupyter from gaia-explorer or gaia-explorer/notebooks.')
sys.path.insert(0, str(ROOT))
from src import classification as data, visualize
# Refresh the data loader if this kernel imported an older version.
data = importlib.reload(data)
visualize = importlib.reload(visualize)  # Pick up updated plot labels and layout.

DATA_PATH = None  # None uses the shared W2 cache; a missing cache is downloaded.
OUTPUT = ROOT / 'data/processed/week3'
OUTPUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)


## Load the Day 1 data

These lines select the saved splits. `X` contains color and `y` contains the brightness label.

In [ ]:
prepared_path = OUTPUT / 'learning_data.csv'
if not prepared_path.is_file():
    raise FileNotFoundError('Run Week 3 Day 1 first to create learning_data.csv.')

# Keep source IDs as text so their digits are not rounded.
gaia = pd.read_csv(prepared_path, dtype={'source_id': 'string'})
train = gaia.loc[gaia['split'] == 'train'].copy()
validation = gaia.loc[gaia['split'] == 'validation'].copy()
test = gaia.loc[gaia['split'] == 'test'].copy()
feature_columns = ['bp_rp']
X_train = train[feature_columns]
y_train = train['is_bright']
X_val = validation[feature_columns]
y_val = validation['is_bright']
print('Training rows:', len(train), '| Validation rows:', len(validation))


## 1. Calculate loss for one star

Loss is −log(p) when y=1 and −log(1−p) when y=0. Expected outputs: loss table and curves. A confidently wrong prediction has a large loss.

In [ ]:
p = np.array([0.01, 0.5, 0.99])
loss_when_positive = -np.log(p)
loss_when_negative = -np.log(1 - p)
display(pd.DataFrame({'probability': p, 'loss_if_y_1': loss_when_positive, 'loss_if_y_0': loss_when_negative}))
p = np.linspace(0.001, 0.999, 300)
visualize.plot_binary_curves(p, {'y=1': -np.log(p), 'y=0': -np.log1p(-p)}, xlabel='Probability', ylabel='Loss', title='Logistic loss')

## 2. Define the cost function

Average the loss across all rows. `lambda_=0` gives ordinary logistic cost. A positive lambda adds λΣw²/(2m), penalizing weights but not bias. Expected output: a function whose calculation you can inspect.

In [ ]:
def compute_cost(X, y, w, b, lambda_=0.0):
    """Mean logistic loss plus L2 regularization; bias b is not penalized."""
    m = len(y)
    z = X @ w + b
    # Stable form of -y*log(sigmoid(z)) - (1-y)*log(1-sigmoid(z)).
    loss = np.mean(np.logaddexp(0, z) - y * z)
    regularization = lambda_ * np.sum(w ** 2) / (2 * m)
    return loss + regularization


## 3. Check the initial cost

Before training, we set every weight and the intercept to **zero**. This gives us a simple answer that we can calculate by hand, so we can check `compute_cost`.

- `scaler.fit_transform(X_train)` learns the mean and standard deviation of the **training colors** and scales them. `X` contains those scaled BP-RP values. Scaling prepares the data for training later; with zero weights, color does not affect the initial prediction.
- `y_train.to_numpy()` makes `y`, an array of the known 0/1 brightness labels.
- `X.shape[1]` counts the input columns. There is one input column (`bp_rp`), so `np.zeros(X.shape[1])` makes `w = [0]`. We also set the intercept `b = 0`.

For **every** training star, the score and probability are now:

```text
score = w × scaled_color + b = 0
probability = sigmoid(0) = 1 / (1 + exp(0)) = 0.5
```

Logistic loss is the negative logarithm of the probability assigned to the *correct* label. Whether the true label is 1 or 0, a prediction of 0.5 gives:

```text
true label 1: loss = -log(0.5)
true label 0: loss = -log(1 - 0.5) = -log(0.5)
-log(0.5) = -log(1/2) = log(2) ≈ 0.693
```

`np.log` is the natural logarithm. If each star has loss `log(2)`, their **average cost** is also `log(2)`. The two printed numbers below should therefore match. This check does not train the model; it checks the cost function at its starting values.


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X_train)
y = y_train.to_numpy()
w = np.zeros(X.shape[1])
b = 0.0
print('Computed cost:', compute_cost(X, y, w, b))
print('Expected at zero weights:', np.log(2))

## 4. Try regularization

Expected output: larger lambda increases cost for fixed nonzero weights. This is the penalty itself; we have not yet retrained the weights.

In [ ]:
w = np.ones(X.shape[1])
for strength in [0, 1, 100]:
    print('lambda:', strength, '| cost:', compute_cost(X, y, w, b, lambda_=strength))

## Reflection questions and answers

1. **Why does zero-weight cost equal `log(2)`?** With `w = 0` and `b = 0`, every star has score 0 and sigmoid probability 0.5. The loss for either true label is `-log(0.5) = log(2) ≈ 0.693`. Since all stars have that loss, their average cost is also `log(2)`.

2. **Why not optimize accuracy directly?** Accuracy looks only at the final label, 0 or 1. For example, probabilities 0.6 and 0.7 both become label 1 at a threshold of 0.5, so accuracy cannot show the improvement. As weights change, accuracy stays flat for a while and then jumps when a probability crosses the threshold. Gradient descent needs a cost that changes smoothly; logistic loss changes as the probabilities change.

3. **Can cost improve without accuracy changing?** Yes. Suppose a star's true label is 1. If its predicted probability rises from 0.6 to 0.9, both predictions still become label 1, so accuracy remains correct. Its loss improves from `-log(0.6) ≈ 0.51` to `-log(0.9) ≈ 0.11`. The average cost can therefore fall even when accuracy stays the same.


## References

Course labs: `../ML-coursework/W3/Optional Labs/` in the workspace.

[ESA Gaia DR3](https://www.cosmos.esa.int/web/gaia/dr3) · [Extinction guidance](https://www.cosmos.esa.int/web/gaia/edr3-extinction-law) · [Scikit-learn logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)